# 01 · Annotation viewer

Shows a squeeze under both lighting rotations with its annotated letter boxes, and how the
transcript rows are grouped into the line crops used to train the CRNN.

Replaces the earlier `final_one.ipynb` and `ctc_dataset_building_and_test.ipynb`.

In [ ]:
import os
from pathlib import Path

# Run from the repository root so the default config paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())

In [ ]:
import cv2
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

from trogs.data.annotations import parse_letters, parse_text_lines
from trogs.data.geometry import line_rect, warp_line

IMG_DIR = "data/raw/Images"
ANN_DIR = "data/raw/Annotations"
SQUEEZE = "IAS_27288"

## Letter boxes on both rotations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 12))
for ax, rotation, color in zip(axes, (1, 2), ("lime", "yellow")):
    base = f"{SQUEEZE}_Rotation{rotation}_300dpi"
    img = cv2.cvtColor(cv2.imread(f"{IMG_DIR}/{base}.png"), cv2.COLOR_BGR2RGB)
    boxes, labels = parse_letters(f"{ANN_DIR}/{base}_letters.txt")
    ax.imshow(img)
    ax.set_title(f"{SQUEEZE} - Rotation {rotation} ({len(boxes)} letters)", fontsize=16)
    ax.axis("off")
    for box, label in zip(boxes, labels):
        ax.add_patch(Polygon(box, closed=True, fill=True, edgecolor=color, facecolor=(0, 1, 0, 0.1), lw=1.2))
        ax.text(box[:, 0].min(), box[:, 1].min(), label, color=color, fontsize=11, weight="bold",
                ha="left", va="bottom", path_effects=[pe.withStroke(linewidth=2, foreground="black")])
plt.tight_layout()
plt.show()

## Line crops for the CRNN

Each transcript row consumes one box per non-`*` character. The crop is the minimum-area rectangle around the row, rectified to height 32.

In [ ]:
base = f"{SQUEEZE}_Rotation1_300dpi"
img = cv2.imread(f"{IMG_DIR}/{base}.png")
lines = parse_text_lines(f"{ANN_DIR}/{base}_letters.txt")
print(f"{len(lines)} lines")

fig, axes = plt.subplots(len(lines), 1, figsize=(14, 1.2 * len(lines)), squeeze=False)
for ax, line in zip(axes[:, 0], lines):
    center, w, h, angle = line_rect(line.points)
    crop = warp_line(img, center, w + h, h, angle, target_h=32, border_value=(0, 0, 0))
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(line.text, loc="left", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()